# **Supported Vector Machines (SVM)**
En este capitulo hablaremos sobre otra familia de algoritmos supervisados, las supported vector machines.  
Antes de empezar a desarrollar este tema fijemosnos en una cosa sobre la regresion lineal. Para recordar, dado un dato de entrada x la regresion lineal seleccionaba unos parametros $\theta$ tal que al multiplicaros por el dato x daban una buena aproximacion de y. En el capitulo anterior el vector x que hemos estado usando es tal cual una observacion del training set es decir $(x_1, x_2, ..., x_m)$. Esto lo que hace construir una relacion lineal entre X e Y. Ahora veamos que en realidad podemos modificar el vector x de entrada para que la funcion que estima nuestro modelo tenga mas grados de libertad. Por ejemplo si intoducimos una caracteristica $x_0$ constantemente 1 estamos dado capacidad al modelo para poder elegir un bias o cambiar la interccepcion con el eje y. Tambien podemos trasnformar x de forma que sea $(x_1, x_2, ..., x_m, x_1^2, x_2^2, ..., x_m^2)$ y entonces la funcion que estimamos ya no es necesariamente una recta sino que pueden ser funciones cuadraticas.  
Es precisamente este concepto el que explotan las supported vector machines. El algoritmo sera similar ha una regresion logistica pero introducira un mapeo del vector x a un vector de dimension superior con mas caracteristicas explotando comportamientos del producto escalar de dos vectores.

## **Kernels** (Kernel Trick)
Este desarrollo sera bastante teorico y tardara un rato hasta que lleguemos a una implementacion de una Supported Vector Machine. Vamos a empezar introduciento el concepto de kernel y el kernel trick.  
Hemos dicho que los que queremos hacer es mapear el vector x a otro de dimension superior con el objetivo de encontrar relaciones mas complejas no lieneales en los datos. Supongamos que mapeamos x a un vector que contiene todos los monomios de orden menos o igual que tres de elementos de x. Esto lo haremos mediante la funcion $\phi : \mathbb{R}^n \to \mathbb{R}^{n^3}$ de forma que nuestra funcion de actualizacion del parametro $\theta$ usando descenso de gradiente queda:
$$
\theta := \theta + \alpha \sum_{i=1}^{n} \left( y^{(i)} - \theta^T \phi(x^{(i)}) \right) \phi(x^{(i)})
$$
El problema aqui es que en cada paso tenemos que si nuestro vector $x^{(i)}$ era de dimension m ahora $\phi(x^{(i)})$ es de dimension $m^3$ y almacenar y hacer calulos con un vector de esas dimensiones es costoso. Parece dificil conseguir un algoritmo que sea mejor ya que $\theta$ es de esas dimensiones. Aqui es donde entra en kernel trick.  

**El kernel trick va a consistir en:  
(1) Demostrar que $\theta$ es combinacion lineal de $\phi(x^{(i)})$ y por tanto lo ponemos representar como n coeficientes $\beta_i$  
(2) Demostrar que la funcion de actualizacion de los parametros se puede expresar en funcion de $\langle \phi(x^{(i)}), \phi(x^{(j)}) \rangle$ en vez de en funcion de $\phi(x^{(i)})$ directamente.  
(3) Demostrar que hay una manera mas eficiente de calcular $\langle \phi(x^{(i)}), \phi(x^{(j)}) \rangle$**

### $\theta$ es combinacion lineal de $\phi(x^{(i)})$
Por induccion:  
Caso inicial:  
$$
\theta = 0 = \sum_{i=1}^{n} \beta_i \phi(x^{(i)}) \quad con \quad \beta_i=0
$$  
Hipostesis de induccion: Asumimos que se cumple para un $\theta$ y veamos que se cumple para el siguiente:  
La regla de actualizacion de $\theta$ es:
$$
\theta := \theta + \alpha \sum_{i=1}^{n} \left( y^{(i)} - \theta^T \phi(x^{(i)}) \right) \phi(x^{(i)})
$$
Por la hipotesis de induccion:
$$
\theta := \sum_{i=1}^{n} \beta_i \phi(x^{(i)}) + \alpha \sum_{i=1}^{n} \left( y^{(i)} - \theta^T \phi(x^{(i)}) \right) \phi(x^{(i)})
$$
$$
\theta := \sum_{i=1}^{n} \left( \beta_i  + \alpha \left( y^{(i)} - \theta^T \phi(x^{(i)}) \right)\right) \phi(x^{(i)})
$$
de donde el termino entre parentesis es un escalar y la demostracion queda concluida.  

### La funcion de actualizacion de los parametros se puede expresar en funcion de $\langle \phi(x^{(i)}), \phi(x^{(j)}) \rangle$
Partimos de la expresion anterior, sustituyendo $\theta = \sum_{i=1}^{n} \beta_i \phi(x^{(i)})$ en las dos $\theta$ que faltan:
$$
\sum_{i=1}^{n} \beta_i \phi(x^{(i)}) := \sum_{i=1}^{n} \left( \beta_i  + \alpha \left( y^{(i)} - \sum_{j=1}^{n} \beta_i \phi(x^{(j)})^T \phi(x^{(i)}) \right)\right) \phi(x^{(i)})
$$
Igualando termino a termino  y dividiendo todo entre $\phi(x^{(i)})$ tenemos:
$$
\beta_i := \beta_i  + \alpha \left( y^{(i)} - \sum_{j=1}^{n} \beta_j \phi(x^{(j)})^T \phi(x^{(i)}) \right) \qquad \forall i = 1, \dots, n
$$
$$
\beta_i := \beta_i  + \alpha \left( y^{(i)} - \sum_{j=1}^{n} \beta_j \langle \phi(x^{(i)}), \phi(x^{(j)}) \rangle \right) \qquad \forall i = 1, \dots, n
$$
De esta forma queda demostrado el punto 2.  

### Hay una manera mas eficiente de calcular $\langle \phi(x^{(i)}), \phi(x^{(j)}) \rangle$
Sean $x, z \in \mathbb{R}^{m}$, entonces por la forma de $\phi$:
$$\langle \phi(x), \phi(z) \rangle = \sum_{i=1}^{m} x_i z_i + \sum_{i,j=1}^{m} x_i x_j z_i z_j+ \sum_{i,j,k=1}^{m} x_i x_j x_k z_i z_j z_k$$
$$=1 + \sum_{i=1}^{d} x_i z_i + \left( \sum_{i=1}^{d} x_i z_i \right)^2+ \left( \sum_{i=1}^{d} x_i z_i \right)^3$$
$$= 1 + \langle x, z \rangle + \langle x, z \rangle^2 + \langle x, z \rangle^3$$
Por lo tanto en realidad solo debemos calcular $\langle x, z \rangle$ con vectores que son $\mathbb{R}^{m}$ y no $\mathbb{R}^{m^3}$. Es decir existe una forma mas eficiente de calcular $\langle \phi(x^{(i)}), \phi(x^{(j)}) \rangle$ que la estandar.  

Con todo este desarrollo que acabamos de hacer es encapsular toda la informacion del mapeo en los productos escalares. **Definimos el kernel asociado al mapeo $\phi$** como ese producto escalar. $K(x, z) = \langle \phi(x), \phi(z) \rangle$ que encapsula toda la informacion del mapeo.  
Si encontramos una caracterizacion de los kernels entonces podemos pasar de elegir un mapeo y calcular su kernel asociado a definir un kernel y comprobar si cumple la caracterizacion y de esta manera que sea innecesario escribir la expresion del mapeo de manera explicita.  
Con propiedades del producto escalar podemos llegar a desmostrar este teorema:  
**Teorema de Mercer:** Sea $K : \mathbb{R}^m \times \mathbb{R}^m \to \mathbb{R}$. Entonces K es un kernel valido (en sentido Mercer) si y solo si para todo $\{x^{(1)}, x^{(2)}, ..., x^{(n)}\} \quad n<\infty$ se tiene que K es simetrica y semi-definida positiva.  
Un ejemplo de kernel es el gausiano que es de la forma:
$$K(x, z) = \exp\Big( - \frac{\|x - z\|^2}{2\sigma^2} \Big)$$
Esta idea de kernels la podemos aplicar el muchos algoritmos como en regresion linel o logistica como hemos visto. Pero el metodo en el que se aplican de manera mas natural es en Supported Vector Machine.

## **Supported Vector Machines**
Estamos ante un problema de clasificaccion. Es decir dada una observacion la queremos clasificar como -1 o 1. Por lo tanto de manera similar a cuando hicimos regresion logistica vamos a definir una funcion predictora:
$$
h_{w, b}(x) = g(w^T x + b)
$$
Aqui g va a ser tal que g(z) = 1 si z>=0 y g(z) = -1 en otro caso y como notacion vamos a cambiar la forma en que denominamos los parametros de $\theta$ a $(w, b)$.  
Esntonces esto lo que hara sera separar el espacio de posibles observaicones mediante una recta ($w^T x + b=0$). A un lado quedaran los $w^T x + b<0$ y a otro los $w^T x + b>=0$.
Se puede intuir que cuanto mas cercana este una observacion menos confianza tenemos sobre como calsificarla por lo tanto vamos a definir una medida que refleje cuanto de lejos esta la observacion del umbral. La distancia entre punto y recta no da que lo que vamos a denominar **margen geometrico** es:
$$
\gamma^{(i)} = y^{(i)} \, \frac{w^T x^{(i)} + b}{\|w\|}
$$
$$
\gamma = \min_{i=1, \dots, n} \gamma^{(i)}
$$
Como idea similar podemos definir el **margen funcional** que a diferencia del geometrico no es invariante frente al reescalado de parametros.
$$
\hat{\gamma^{(i)}} = y^{(i)} \, (w^T x^{(i)} + b)
$$
$$
\hat{\gamma} = \min_{i=1, \dots, n} \hat{\gamma^{(i)}}
$$

Llegados a este punto el problema que estamos tratando de solucionar es obvio. Minimizar el margen geometrico, es decir elegir $(w, b)$ de forma que la distancia de los puntos al hiperplano determinado por $(w, b)$ sea maxima y evidentemente todo esto restringido a que esten en el lado correcto de hiperplano. Matematicamente tenemos que resolver:
$$
\begin{aligned}
\max_{w, b, \gamma} \quad & \gamma \\
\text{s.a.} \quad & y_i \big(w^\top x_i + b\big) \geq \gamma, \quad i = 1, \dots, n, \\
& \|w\| = 1
\end{aligned}
$$
Aqui lo que estamos haciendo maximizar $\gamma$ que es el minimo de los margenes funcionales y añadimos las restriccion de $\|w\| = 1$ para asegurar que el margen funcional coincida con el geometrico. Sin embargo la restriccion $\|w\| = 1$ no es convexa lo que motiva que tratemos de transformar el problema en uno equivalente pero con restricciones y funcion objetivo convexas. De esta forma llegamos a:
$$
\begin{aligned}
\min_{w, b} \quad & \tfrac{1}{2} \|w\|^2 \\
\text{s.a.} \quad & y_i \big(w^\top x_i + b\big) \geq 1, \quad i = 1, \dots, n
\end{aligned}
$$
Ahora bien, principalmente por que nos aparezca la solucion en funcion de productos escalares para poder usar el truco del kernel en vez de resolver este problema lo que haremos sera dualizarlo y resolver el dual. No desarrollare esto con detalle ya que estos contenidos los cubri completamente en la asignatura de Optimizacion pero estan en las notas del curso detallados. Haciendo este desarrollo llegamos a que debemos resolver:
$$
\begin{aligned}
\max_{\alpha} \quad & W(\alpha) = \sum_{i=1}^{n} \alpha_i 
- \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j (x_i^\top x_j) \\
\text{s.a.} \quad & \alpha_i \ge 0, \quad i = 1, \dots, n, \\
& \sum_{i=1}^{n} \alpha_i y_i = 0
\end{aligned}
$$
Aqui los $\alpha$ son los multilicadores de Lagrange asociandos a las restricciones de menor o igual. Si se cumple la igualdad quiere decir que ese ejemplo de entrenamiento esta en la frontera y por tanto la restriccion es activa (su alfa es no nulo). Estos se llaman vectores soporte y la actualizacion de los paramtros depende solo de ellos. Resolviendo este problema llegamos a:
$$
\begin{aligned}
w^\top x + b 
&= \left( \sum_{i=1}^{n} \alpha_i y_i x_i \right)^\top x + b \\
&= \sum_{i=1}^{n} \alpha_i y_i \langle x_i, x \rangle + b
\end{aligned}
$$
Donde vemos usando el Kernel Trick visto antes cambiamos el producto escalar por una funcion K que sea un kernel valido y llegamos a:
$$
w^\top x + b = \sum_{i=1}^{n} \alpha_i y_i K(x_i, x) + b
$$
Y por los tanto llegamos que finalmente nuestra funcion de hipotesis sera:
$$
h_\theta(x_i) = sign \left( \sum_{j=1}^{n} \alpha_j y_j K(x_j, x_i) + b \right)
$$
Primero observamos que la funcion hipotesis solo depende de los puntos en que $\alpha_j \neq 0$ (las restricciones activas). Solo depende de los vectores soporte de hay el nombre.
Como en los vecotres soporte se cumple la igualdad, de ahi despejamos el valor de b:
$$
y_k \big( w^\top x_k + b \big) = 1, \quad \forall (x_k, y_k) \in SV
\quad \Rightarrow \quad
b_k = y_k - \sum_{i \in SV} \alpha_i y_i x_i^\top x_k
$$
$$
b = \frac{1}{|SV|} \sum_{k \in SV} b_k
$$
Finalmente notamos que auque aqui no lo hagamos es posible que los datos sean no reparables o por regularizaciojn queramos dar un margen de error. Esto se puede conseguir modificando un poco el problema pero la idea es la misma.

## **Implementacion**
A diferencia de los otros algoritmos este no lo voy a implementar manualmente si no que lo hare usando sklearn, por variar (y porque aunque no este desarrollado con todo detalle le he dedicado mucho tiempo a estudiar los fundamentos matematicos y estoy un poco cansado).

In [1]:
from sklearn import datasets
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Creamos un dataset para trabajar con el
X, y = datasets.make_classification(n_samples=1000, n_features=2, n_informative=2, n_redundant=0)
y = 2*y - 1  #la labels son {0, 1} por lo que las trasnformamos a {-1, 1} que es lo que nos interesa en SVM

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#Creamos el modelo
SVM = SVC(kernel='linear', C=1.0) #Este C es lo que controla el margen de error que no hemos desarrollado arriba

#Entrenamos el modelo
SVM.fit(X_train, y_train)

#Se lo aplicamos a los datos de test
y_pred = SVM.predict(X_test)

#Evaluamos el modelo
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.945
